# Airline Passenger Satisfaction

Lab 1 notebook, structured the same way as the Telco Churn project.
The target is binary: **1 = satisfied**, **0 = neutral or dissatisfied**.

In [ ]:
import os
import json
import joblib
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')

directories = [
    '../data/raw', '../data/processed', '../notebooks', '../src',
    '../models', '../logs', '../outputs', '../artifacts'
]
for directory in directories:
    os.makedirs(directory, exist_ok=True)

print('Project directory structure initialized!')

## 1. Load and Inspect Dataset

In [ ]:
df = pd.read_csv('../data/raw/airline.csv')
print('Dataset loaded successfully!')
print('Shape:', df.shape)
df.head()

## 2. Preprocessing & Data Cleaning

Drop identifier columns, fill missing arrival delay with the median, and encode the target.

In [ ]:
df = df.drop(columns=['Unnamed: 0', 'id'], errors='ignore')
df['Arrival Delay in Minutes'] = pd.to_numeric(df['Arrival Delay in Minutes'], errors='coerce')
missing_delay = int(df['Arrival Delay in Minutes'].isna().sum())
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())
print('Missing arrival delays filled:', missing_delay)

df['satisfaction'] = df['satisfaction'].apply(lambda x: 1 if str(x).strip().lower() == 'satisfied' else 0).astype(int)
print(df['satisfaction'].value_counts())

X = df.drop('satisfaction', axis=1)
y = df['satisfaction']

## 3. Train-Test Split, Feature Engineering & Artifact Export

Split first, then scale numeric columns and one-hot encode categorical columns.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = X_train.select_dtypes(include=['object', 'category']).columns
num_cols = X_train.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train[num_cols])
x_test_scaled = scaler.transform(X_test[num_cols])

ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
x_train_encoded = ohe.fit_transform(X_train[cat_cols])
x_test_encoded = ohe.transform(X_test[cat_cols])

X_train_final = np.hstack((x_train_scaled, x_train_encoded))
X_test_final = np.hstack((x_test_scaled, x_test_encoded))

np.save('../data/processed/X_train_final.npy', X_train_final)
np.save('../data/processed/X_test_final.npy', X_test_final)
np.save('../data/processed/y_train.npy', y_train.to_numpy(dtype=np.int64))
np.save('../data/processed/y_test.npy', y_test.to_numpy(dtype=np.int64))

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(ohe, '../models/ohe.pkl')

metadata = {
    'dataset_name': 'Airline Passenger Satisfaction',
    'train_shape': list(X_train_final.shape),
    'test_shape': list(X_test_final.shape),
    'numerical_features': list(num_cols),
    'categorical_features': list(cat_cols),
    'target': 'satisfaction',
    'positive_class': 'satisfied'
}
with open('../data/processed/dataset_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print('Data split, engineered, and saved with scaler, encoder, and metadata!')
print('Train shape:', X_train_final.shape, 'Test shape:', X_test_final.shape)

## 4. Baseline Model Development

Train Logistic Regression, Decision Tree, and Random Forest with balanced class weights.

In [ ]:
def evaluate_and_store_metrics(model, name, X_test, y_test, results_dict, roc_dict):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f'--- {name} ---')
    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'F1-Score : {f1:.4f}\n')

    results_dict[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_dict[name] = {'fpr': fpr, 'tpr': tpr, 'auc': auc(fpr, tpr)}

## 5. Model Evaluation

Models are compared with accuracy, precision, recall, F1, and ROC-AUC.
The model with the highest **recall** for satisfied passengers is saved, and false positives / false negatives are exported.

In [ ]:
metrics_results = {}
roc_data = {}

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
}

best_recall = -1
best_model_name = ''
best_model = None

for model_name, model in models.items():
    model.fit(X_train_final, y_train)
    evaluate_and_store_metrics(model, model_name, X_test_final, y_test, metrics_results, roc_data)
    current_recall = metrics_results[model_name]['Recall']
    if current_recall > best_recall:
        best_recall = current_recall
        best_model_name = model_name
        best_model = model

print('==========================================')
print(f'Best Model: {best_model_name} (Recall: {best_recall:.4f})')
print('==========================================')

file_name = best_model_name.replace(' ', '_').lower()
joblib.dump(best_model, f'../models/{file_name}_baseline.pkl')

predictions = best_model.predict(X_test_final)
errors_df = X_test.copy()
errors_df['Actual_Satisfaction'] = y_test.values
errors_df['Predicted_Satisfaction'] = predictions

false_negatives = errors_df[(errors_df['Actual_Satisfaction'] == 1) & (errors_df['Predicted_Satisfaction'] == 0)]
false_positives = errors_df[(errors_df['Actual_Satisfaction'] == 0) & (errors_df['Predicted_Satisfaction'] == 1)]
false_negatives.to_csv('../outputs/false_negatives.csv', index=False)
false_positives.to_csv('../outputs/false_positives.csv', index=False)
print('Error analysis saved to outputs/.')

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=['neutral or dissatisfied', 'satisfied'],
    cmap='Blues'
)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.show()

## 6. ROC-AUC Curve Analysis

The ROC curve compares how well each model separates satisfied passengers from neutral or dissatisfied passengers. An AUC closer to 1 is stronger.

In [ ]:
plt.figure(figsize=(7, 5))
for name, curve in roc_data.items():
    plt.plot(curve['fpr'], curve['tpr'], label=f"{name} (AUC = {curve['auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Airline Passenger Satisfaction')
plt.legend()
plt.tight_layout()
plt.show()